# Exploratory Data Analysis - Superstore Dataset

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

%matplotlib inline
sns.set(style="whitegrid")
warnings.filterwarnings('ignore')

# Load the dataset
df = pd.read_csv('superstore.csv')

ModuleNotFoundError: No module named 'pandas'

## Q1: Data Cleaning

In [ ]:
# i. Handling Missing values
df = df.dropna(subset=['Shipping.Cost', 'State'])

# ii. Duplicate values
df = df.drop_duplicates()

# iii. Delete unknown columns
if '记录数' in df.columns:
    df = df.drop(columns=['记录数'])

# Standardizing column names
df.columns = [col.replace('.', '_').lower() for col in df.columns]

print(f"Cleaned Shape: {df.shape}")
df.head()

**Insights Q1:**
- Missing values in Shipping Cost and State were removed.
- The unknown column '记录数' was deleted.
- Column names were formatted to lowercase with underscores for better coding consistency.

## Q2: Univariate Analysis - Numerical Features

In [ ]:
num_features = ['discount', 'profit', 'quantity', 'sales', 'shipping_cost']

plt.figure(figsize=(15, 5))
for i, col in enumerate(num_features):
    plt.subplot(1, 5, i+1)
    sns.histplot(df[col], kde=True)
    plt.title(f'Dist: {col}')
plt.tight_layout()
plt.show()

print("Skewness:\n", df[num_features].skew())

**Insights Q2:**
- **i. Useless Features:** `row_id` and `order_id` are identifiers and not useful for distribution analysis.
- **ii. Normality:** None of the features follow a normal distribution.
- **iii. Skewness:** Sales (8.14) and Profit (4.16) are highly right-skewed, indicating a few very high-value orders.
- **iv. Outliers:** Significant outliers exist in Profit and Sales, meaning the median is a better measure of central tendency than the mean.

## Q3: Univariate Analysis - Categorical Features

In [ ]:
plt.figure(figsize=(10, 6))
sns.countplot(data=df, x='category', order=df['category'].value_counts().index)
plt.title('Order Count by Category')
plt.show()

print(f"Unique Cities: {df['city'].nunique()}")
print(f"Top Country: {df['country'].value_counts().idxmax()}")

**Insights Q3:**
- **ii. Customer Name:** Using this as a category is bad for modeling due to high cardinality (795 unique names).
- **iii. Category Distribution:** Skewed towards Office Supplies because they are low-cost, high-frequency essentials.
- **iv. Geographic Bias:** The United States dominates the data, making it the primary driver of the dataset's trends.

## Q4: Bivariate Analysis - Numerical to Numerical

In [ ]:
plt.figure(figsize=(8, 6))
sns.heatmap(df[num_features].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix')
plt.show()

**Insights Q4:**
- **i. Strongest Correlation:** Sales and Shipping Cost (0.77).
- **ii. Negative Correlation:** Discount and Profit (-0.32).
- **iv. Time Effects:** Sales have increased steadily year-over-year from 2011 to 2014.

## Q5: Bivariate Analysis - Categorical to Numerical

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 6))
sns.boxplot(data=df, x='category', y='profit', ax=ax[0])
ax[0].set_ylim(-200, 200)
ax[0].set_title('Profit by Category')

sns.barplot(data=df, x='category', y='sales', estimator=np.median, ax=ax[1])
ax[1].set_title('Median Sales by Category')
plt.show()

**Insights Q5:**
- **ii. Sales/Profit by Category:** Technology has both the highest median sales and median profit.
- **iii. Profit by Segment:** Home Office has the highest median profit. Consumer segment has the most extreme negative outliers.

## Q6: Market Analysis

In [ ]:
print("Market vs Region Map:\n", pd.crosstab(df['market'], df['region']))

office_supplies = df[df['category'] == 'Office Supplies']
print("\nCountries with 1 Office Supply order:", 
      office_supplies['country'].value_counts()[office_supplies['country'].value_counts() == 1].index.tolist())

**Insights Q6:**
- **i. Market Spread:** Markets are not random; they are strictly aligned with specific geographic regions.
- **ii. Negligible Orders:** Countries like Swaziland and Armenia have only 1 office supply order.
- **iii. Business Insight:** Strategy should focus on Technology for margins and US/APAC markets for volume.